In [3]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor

ruta = '/home/jovyan/work/autotec/trabajoneiel/Semana 15/datos_automotriz_dashboard.csv'
df = pd.read_csv(ruta)

objetivo = 'precio'
df = df.dropna(subset=[objetivo]).copy()

X = df.drop(columns=[objetivo])
y = df[objetivo]

num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = X.select_dtypes(exclude='number').columns.tolist()

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_cols),
        ('cat', categorical_transformer, cat_cols)
    ]
)

modelo = RandomForestRegressor(random_state=42, n_jobs=-1)

pipe = Pipeline([
    ('prep', preprocessor),
    ('model', modelo)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

param_grid = {
    'model__n_estimators': [200, 300],
    'model__max_depth': [None, 10, 20],
    'model__min_samples_split': [2, 5],
    'model__min_samples_leaf': [1, 2]
}

grid = GridSearchCV(
    pipe,
    param_grid=param_grid,
    cv=3,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=1,
    error_score='raise'
)

grid.fit(X_train, y_train)
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)
r2 = r2_score(y_test, y_pred)

print('Mejores parámetros:', grid.best_params_)
print('MAE:', mae)
print('RMSE:', rmse)
print('R2:', r2)

resultados = pd.DataFrame({'y_real': y_test.values, 'y_predicho': y_pred})
resultados.to_csv('resultados_prediccion_precio.csv', index=False)

metricas = pd.DataFrame([{
    'modelo': 'RandomForestRegressor',
    'mae': mae,
    'rmse': rmse,
    'r2': r2,
    'mejores_parametros': str(grid.best_params_)
}])
metricas.to_csv('metricas_modelo_precio.csv', index=False)

joblib.dump(best_model, 'modelo_precio_autotec.joblib')

Fitting 3 folds for each of 24 candidates, totalling 72 fits
Mejores parámetros: {'model__max_depth': None, 'model__min_samples_leaf': 1, 'model__min_samples_split': 2, 'model__n_estimators': 200}
MAE: 2768339.0854271357
RMSE: 4893661.213834106
R2: 0.7384935433441373


['modelo_precio_autotec.joblib']

## Resultado del Modelo

In [4]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)
r2 = r2_score(y_test, y_pred)

# "Accuracy" aproximada para regresión:
# porcentaje de predicciones dentro de un margen de error tolerable
margen = 0.10  # 10%
accuracy_regresion = np.mean(np.abs(y_test - y_pred) / y_test <= margen)

print(f"Coeficiente de determinación (R2): {r2:.4f}")
print(f"Error absoluto medio (MAE): {mae:,.2f}")
print(f"Raíz del error cuadrático medio (RMSE): {rmse:,.2f}")
print(f"Accuracy aproximado con margen del {margen*100:.0f}%: {accuracy_regresion*100:.2f}%")

evaluacion = pd.DataFrame([{
    "R2": r2,
    "MAE": mae,
    "RMSE": rmse,
    "Accuracy_aproximado": accuracy_regresion
}])

evaluacion.to_csv("evaluacion_modelo_regresion.csv", index=False)

Coeficiente de determinación (R2): 0.7385
Error absoluto medio (MAE): 2,768,339.09
Raíz del error cuadrático medio (RMSE): 4,893,661.21
Accuracy aproximado con margen del 10%: 41.46%
